# Notebook 4 - Incremental Update Pipeline (New_Dataset only)
### Rancang Bangun Chatbot Sumber Informasi Sivitas UPI Berbasis RAG

**Purpose:** Process only the new files in `Dataset\New_Dataset` and **append**
their chunks + vectors to the existing pipeline outputs. Does NOT touch the
original 2076 documents.

| Stage | Reads | Writes |
|-------|-------|--------|
| 1 Discover  | `New_Dataset/**/*.pdf` minus existing `manifest.json` ids | in-memory list of new docs |
| 2 Extract   | new PDF/TXT/HTML files | `raw/<id>.json` (new only) |
| 3 Clean+MD  | new raw json | `cleaned/<id>.{json,txt,md}` (new only) |
| 4 Chunk     | new cleaned json | appends to `chunked/chunks.jsonl` |
| 5 Embed     | new chunks | shard `.npy` + `faiss.index.add()` + `chunks_meta.json` append |

**Idempotent**: re-run as many times as you want — files already processed are
detected by `doc_id` and skipped at every stage.

**Requirement**: Notebooks 1-3 must have completed at least once so that
`manifest.json`, `chunks.jsonl`, `faiss.index`, and `chunks_meta.json` exist.


## Cell 1 - Imports, paths, config

In [1]:
# =============================================================================
# Notebook 4 - Imports & config
# =============================================================================
import json, re, os, io, time, hashlib, logging, traceback, statistics
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
from tqdm.auto import tqdm

# --- Paths (match Notebooks 1-3) ---
DRIVE_BASE = Path(os.environ.get("RAG_UPI_BASE", r"D:\Project\RAG_UPI\Dataset"))
NEW_DATASET_ROOT = DRIVE_BASE / "New_Dataset"   # ONLY source for this notebook
PIPE = DRIVE_BASE / "_pipeline"

DIRS = {
    "raw":     PIPE / "raw",
    "cleaned": PIPE / "cleaned",
    "chunked": PIPE / "chunked",
    "index":   PIPE / "index",
    "logs":    PIPE / "logs",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = PIPE / "manifest.json"
CHUNKS_JSONL  = DIRS["chunked"] / "chunks.jsonl"
INDEX_FILE    = DIRS["index"] / "faiss.index"
META_FILE     = DIRS["index"] / "chunks_meta.json"
INFO_FILE     = DIRS["index"] / "index_info.json"
SHARD_DIR     = DIRS["index"] / "shards"
SHARD_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "doc_extensions": [".pdf", ".html", ".htm", ".txt"],
    # OCR
    "ocr_languages_paddle": "id",
    "ocr_languages_tesseract": "ind+eng",
    "ocr_min_chars_per_page": 80,
    "ocr_dpi": 200,
    "ocr_engine_preference": "tesseract",
    # Markdown
    "markdown_heading_size_factor": 1.15,
    "markdown_max_heading_level": 3,
    "markdown_front_matter": True,
    # Chunking
    "chunk_size": 1000,
    "chunk_overlap": 200,
    "min_chunk_chars": 50,
    # Embedding — MUST match what built the existing FAISS index
    "embedding_model": "intfloat/multilingual-e5-base",
    "use_e5_prefixes": True,
    "embedding_batch_size": 32,
}

# --- Logging ---
def get_logger(name="nb4", logfile=None):
    logger = logging.getLogger(name); logger.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s",
                            datefmt="%H:%M:%S")
    if not any(isinstance(h, logging.StreamHandler) for h in logger.handlers):
        sh = logging.StreamHandler(); sh.setFormatter(fmt); logger.addHandler(sh)
    if logfile and not any(isinstance(h, logging.FileHandler) for h in logger.handlers):
        try:
            fh = logging.FileHandler(logfile, encoding="utf-8")
            fh.setFormatter(fmt); logger.addHandler(fh)
        except Exception: pass
    logger.propagate = False
    return logger

log = get_logger("nb4", logfile=str(DIRS["logs"] / "nb4_update.log"))


# --- Shared helpers ---
def doc_id(source):
    """Stable 16-char id; matches N1's id function."""
    return hashlib.sha1(str(source).encode("utf-8")).hexdigest()[:16]


def read_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    try: return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError: return default


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)


def write_text_atomic(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)


def category_of(source_path):
    """Infer category from immediate subfolder under New_Dataset/."""
    try:
        rel = Path(source_path).resolve().relative_to(NEW_DATASET_ROOT.resolve())
        parts = rel.parts
        return parts[0] if len(parts) > 1 else "New_Dataset"
    except Exception:
        return "New_Dataset"


log.info("Base: %s", DRIVE_BASE)
log.info("New_Dataset root: %s", NEW_DATASET_ROOT)
log.info("Pipeline: %s", PIPE)

# --- Sanity checks: notebook 1-3 outputs must exist ---
if not MANIFEST_PATH.exists():
    log.warning("manifest.json missing — Notebook 1 has not been run.")
if not CHUNKS_JSONL.exists():
    log.warning("chunks.jsonl missing — Notebook 2 has not been run.")
if not INDEX_FILE.exists():
    log.warning("faiss.index missing — Notebook 3 has not finished. "
                "Stage 5 (embedding) will skip writing to FAISS until it exists.")


01:50:17 | INFO    | Base: D:\Project\RAG_UPI\Dataset
01:50:17 | INFO    | New_Dataset root: D:\Project\RAG_UPI\Dataset\New_Dataset
01:50:17 | INFO    | Pipeline: D:\Project\RAG_UPI\Dataset\_pipeline


## Cell 2 - Stage 1: Discover only NEW files in New_Dataset

Recursively walks `New_Dataset/`, computes the stable `doc_id` for every
candidate, and filters out anything already recorded in the master
`manifest.json`. Result is the list of files that need processing.

In [2]:
# =============================================================================
# Stage 1: Discover new files
# =============================================================================
def scan_new_dataset():
    """Recursively collect supported documents under New_Dataset/."""
    exts = set(CONFIG["doc_extensions"])
    if not NEW_DATASET_ROOT.exists():
        log.error("New_Dataset root missing: %s", NEW_DATASET_ROOT)
        return []
    items, seen = [], set()
    for path in NEW_DATASET_ROOT.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in exts:
            continue
        if str(path).startswith(str(PIPE)):
            continue
        key = str(path.resolve())
        if key in seen: continue
        seen.add(key)
        items.append({
            "id": doc_id(key),
            "source": str(path),
            "origin": "local",
            "title": path.stem,
            "type": path.suffix.lower().lstrip("."),
            "category": category_of(path),
            "status": "discovered",
        })
    return items


def filter_new_documents():
    """Return rows from scan_new_dataset() whose id is not already in manifest."""
    candidates = scan_new_dataset()
    existing = {r["id"] for r in read_json(MANIFEST_PATH, default=[])}
    new_rows = [r for r in candidates if r["id"] not in existing]
    by_cat = Counter(r["category"] for r in new_rows)
    log.info("Discovered %d files under New_Dataset, %d already known, "
             "%d truly new.",
             len(candidates), len(candidates) - len(new_rows), len(new_rows))
    if new_rows:
        log.info("New files by category: %s", dict(by_cat))
    return new_rows


# --- RUN ---
new_documents = filter_new_documents()


01:50:20 | INFO    | Discovered 21 files under New_Dataset, 2 already known, 19 truly new.
01:50:20 | INFO    | New files by category: {'Dit-Pendidikan-UPI': 19}


## Cell 3 - Stage 2: Extract new files (PDF text + OCR fallback)

Same extraction logic as Notebook 1, applied only to new documents.
Tesseract auto-located on Windows. Captures per-line font spans for heading
inference.

In [3]:
# =============================================================================
# Stage 2: Extract new files
# =============================================================================
import shutil

# PaddlePaddle 3.x oneDNN workaround (in case Paddle is used)
os.environ.setdefault("FLAGS_use_mkldnn", "0")
os.environ.setdefault("FLAGS_enable_pir_in_executor", "0")

import fitz  # PyMuPDF

_OCR_ENGINE = "uninitialised"
_paddle_ocr = None


def _locate_tesseract():
    try: import pytesseract
    except Exception: return None
    found = shutil.which("tesseract")
    if found:
        pytesseract.pytesseract.tesseract_cmd = found
        return found
    for c in (
        r"C:\Program Files\Tesseract-OCR\tesseract.exe",
        r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
        os.path.expandvars(r"%LOCALAPPDATA%\Programs\Tesseract-OCR\tesseract.exe"),
    ):
        if c and Path(c).exists():
            pytesseract.pytesseract.tesseract_cmd = c
            os.environ["PATH"] = str(Path(c).parent) + os.pathsep + os.environ.get("PATH", "")
            return c
    return None


def _init_ocr():
    global _OCR_ENGINE, _paddle_ocr
    if _OCR_ENGINE != "uninitialised": return _OCR_ENGINE
    pref = CONFIG.get("ocr_engine_preference", "auto")
    if pref == "tesseract":
        try:
            import pytesseract  # noqa
            t = _locate_tesseract()
            if t:
                _OCR_ENGINE = "tesseract"
                log.info("OCR: Tesseract at %s", t)
                return _OCR_ENGINE
        except Exception: pass
    try:
        from paddleocr import PaddleOCR
        for kw in (dict(use_textline_orientation=True, lang=CONFIG["ocr_languages_paddle"]),
                   dict(use_angle_cls=True, lang=CONFIG["ocr_languages_paddle"])):
            try:
                _paddle_ocr = PaddleOCR(**kw)
                _OCR_ENGINE = "paddle"; log.info("OCR: PaddleOCR"); return _OCR_ENGINE
            except Exception: continue
    except Exception: pass
    try:
        import pytesseract  # noqa
        t = _locate_tesseract()
        if t:
            _OCR_ENGINE = "tesseract"; log.info("OCR: Tesseract at %s", t); return _OCR_ENGINE
    except Exception: pass
    _OCR_ENGINE = "none"
    log.error("No OCR engine available. Scanned pages will be empty.")
    return _OCR_ENGINE


def _ocr_image(pil_image):
    engine = _init_ocr()
    if engine == "tesseract":
        import pytesseract
        return pytesseract.image_to_string(pil_image, lang=CONFIG["ocr_languages_tesseract"])
    if engine == "paddle":
        result = _paddle_ocr.ocr(np.array(pil_image)) if hasattr(_paddle_ocr, "ocr") else _paddle_ocr.predict(np.array(pil_image))
        lines = []
        for page in (result or []):
            if isinstance(page, dict) and "rec_texts" in page:
                lines.extend(t for t in page["rec_texts"] if t); continue
            if isinstance(page, (list, tuple)):
                for entry in page:
                    if entry and len(entry) >= 2 and entry[1]:
                        txt = entry[1][0] if isinstance(entry[1], (list, tuple)) else entry[1]
                        if txt: lines.append(txt)
        return "\n".join(lines)
    return ""


def _extract_pdf_layout(page):
    plain = page.get_text("text").strip()
    spans = []
    try:
        d = page.get_text("dict")
        for block in d.get("blocks", []):
            if block.get("type", 0) != 0: continue
            for line in block.get("lines", []):
                line_text = "".join(s.get("text", "") for s in line.get("spans", [])).strip()
                if not line_text: continue
                sizes = [s.get("size", 0) for s in line.get("spans", [])]
                flags = [s.get("flags", 0) for s in line.get("spans", [])]
                bold = any(f & 16 for f in flags)
                size = max(sizes) if sizes else 0
                y0 = block.get("bbox", [0,0,0,0])[1]
                spans.append({"text": line_text, "size": round(size, 2),
                              "bold": bool(bold), "y": round(y0, 1)})
    except Exception: pass
    return plain, spans


def extract_pdf(path):
    from PIL import Image
    pages = []
    doc = fitz.open(path)
    try:
        for i, page in enumerate(doc):
            text, spans = _extract_pdf_layout(page)
            method = "text"
            if len(text) < CONFIG["ocr_min_chars_per_page"]:
                try:
                    pix = page.get_pixmap(dpi=CONFIG["ocr_dpi"])
                    img = Image.open(io.BytesIO(pix.tobytes("png")))
                    ocr_text = _ocr_image(img).strip()
                    if len(ocr_text) > len(text):
                        text, method, spans = ocr_text, "ocr", []
                except Exception as e:
                    log.warning("OCR failed %s p%d: %s", Path(path).name, i+1, e)
            pages.append({"page": i+1, "text": text, "method": method,
                          "n_chars": len(text), "spans": spans})
    finally:
        doc.close()
    return pages


def extract_html(path):
    raw = Path(path).read_text(encoding="utf-8", errors="ignore")
    text = ""
    try:
        import trafilatura
        text = trafilatura.extract(raw, include_comments=False,
                                    include_tables=True, favor_recall=True) or ""
    except Exception: pass
    if len(text.strip()) < 100:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(raw, "lxml")
        for tag in soup(["nav","header","footer","script","style","aside","form","noscript"]):
            tag.decompose()
        main = soup.find("main") or soup.find("article") or soup.body or soup
        text = main.get_text("\n") if main else ""
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return [{"page": 1, "text": text, "method": "html", "n_chars": len(text), "spans": []}]


def extract_one(row):
    src = Path(row["source"])
    ext = src.suffix.lower()
    if ext == ".pdf":
        pages = extract_pdf(src); st = "pdf"
    elif ext in (".html", ".htm"):
        pages = extract_html(src); st = "html"
    elif ext == ".txt":
        t = src.read_text(encoding="utf-8", errors="ignore").strip()
        pages = [{"page": 1, "text": t, "method": "text", "n_chars": len(t), "spans": []}]
        st = "txt"
    else:
        return None
    return pages, st


def run_extraction(new_rows):
    _init_ocr()
    if not new_rows:
        log.info("Nothing new to extract."); return []
    extracted = []
    for row in tqdm(new_rows, desc="Extracting new", unit="doc"):
        raw_path = DIRS["raw"] / f"{row['id']}.json"
        if raw_path.exists():
            extracted.append(row); continue
        try:
            res = extract_one(row)
            if res is None:
                log.warning("Unsupported: %s", row["source"]); continue
            pages, source_type = res
            total = sum(p["n_chars"] for p in pages)
            if total == 0:
                log.warning("Empty extraction: %s", row["source"]); continue
            write_json(raw_path, {
                "id": row["id"], "source": row["source"], "url": None,
                "title": row["title"], "category": row["category"],
                "source_type": source_type, "n_pages": len(pages),
                "ocr_pages": sum(1 for p in pages if p["method"]=="ocr"),
                "total_chars": total, "pages": pages,
                "extracted_at": datetime.now().isoformat(timespec="seconds"),
            })
            extracted.append(row)
        except Exception as e:
            log.warning("Extract failed %s: %s", row["source"], e)
    return extracted


# --- Also append the new rows to manifest.json so we never reprocess them ---
def update_manifest(new_rows):
    existing = {r["id"]: r for r in read_json(MANIFEST_PATH, default=[])}
    for r in new_rows:
        r2 = dict(r); r2["status"] = "extracted"
        existing[r["id"]] = r2
    write_json(MANIFEST_PATH, list(existing.values()))


# --- RUN ---
extracted = run_extraction(new_documents)
update_manifest(extracted)
log.info("Extracted %d new documents.", len(extracted))


01:50:30 | INFO    | OCR: Tesseract at C:\Program Files\Tesseract-OCR\tesseract.exe


Extracting new:   0%|          | 0/19 [00:00<?, ?doc/s]

01:51:06 | INFO    | Extracted 19 new documents.


## Cell 4 - Stage 3: Clean + emit .json/.txt/.md for new docs

In [4]:
# =============================================================================
# Stage 3: Clean + Markdown generation (only for new docs)
# =============================================================================
PAGE_NUM_RE   = re.compile(r"^\s*(?:hal(?:aman)?\.?\s*)?[-]?\s*\d{1,4}\s*(?:/\s*\d{1,4})?\s*[-]?\s*$", re.I)
BULLET_RE     = re.compile(r"^\s*([\-\*o]|\d+[\.\)]|[a-z][\.\)])\s+", re.I)
NUM_LIST_RE   = re.compile(r"^\s*(\d+)[\.\)]\s+(.*)$")
MULTISPACE_RE = re.compile(r"[ \t]{2,}")
OCR_NOISE_RE  = re.compile(r"(.)\1{4,}")
HEADING_RE    = re.compile(r"^([A-Z0-9][A-Z0-9 .,:\'\-/()]{3,80})$")
URL_RE        = re.compile(r"https?://\S+")
BOILERPLATE = {
    "beranda","kontak","login","logout","menu","search","cari",
    "copyright","hak cipta dilindungi","all rights reserved",
    "universitas pendidikan indonesia","bagikan","share","read more",
    "selengkapnya","previous","next","sebelumnya","selanjutnya",
}


def _detect_repeated_lines(pages, min_ratio=0.4):
    counts, n = Counter(), max(len(pages), 1)
    for p in pages:
        for ln in {x.strip() for x in p["text"].splitlines() if x.strip()}:
            if 3 <= len(ln) <= 90: counts[ln] += 1
    return {ln for ln, c in counts.items() if n > 2 and c / n >= min_ratio}


def _is_heading(line):
    s = line.strip()
    return bool(HEADING_RE.match(s)) and len(s.split()) <= 12


def _is_boilerplate(line):
    return line.strip().lower() in BOILERPLATE


def clean_text(text, repeated):
    out = []
    for raw_line in text.splitlines():
        s = raw_line.strip()
        if not s: out.append(""); continue
        if s in repeated or PAGE_NUM_RE.match(s) or _is_boilerplate(s): continue
        s = OCR_NOISE_RE.sub(r"\1", s)
        s = URL_RE.sub("", s).strip()
        s = MULTISPACE_RE.sub(" ", s)
        if not s: continue
        m = NUM_LIST_RE.match(s)
        if m: out.append(f"{m.group(1)}. {m.group(2).strip()}")
        elif BULLET_RE.match(s): out.append("- " + BULLET_RE.sub("", s).strip())
        elif _is_heading(s): out.append("\n## " + s + "\n")
        else: out.append(s)
    return re.sub(r"\n{3,}", "\n\n", "\n".join(out)).strip()


def _infer_heading_levels(pages, factor, max_level):
    sizes = [s["size"] for p in pages for s in (p.get("spans") or []) if s.get("size")]
    if not sizes: return {}
    body = statistics.median(sizes)
    threshold = body * factor
    big = sorted({sp["size"] for p in pages for sp in (p.get("spans") or [])
                  if sp["size"] >= threshold}, reverse=True)
    size_to_level = {sz: min(i+1, max_level) for i, sz in enumerate(big)}
    headings = {}
    for p in pages:
        for sp in p.get("spans") or []:
            if sp["size"] in size_to_level:
                k = sp["text"].strip().lower()
                if 3 <= len(k) <= 120: headings[k] = size_to_level[sp["size"]]
    return headings


def _front_matter(doc):
    fm = {"id":doc["id"],"title":doc["title"],"category":doc["category"],
          "source_type":doc["source_type"],"n_pages":doc["n_pages"],"source":doc["source"]}
    lines = ["---"]
    for k,v in fm.items():
        lines.append(f'{k}: "' + str(v).replace('"',"'") + '"')
    lines.append("---\n")
    return "\n".join(lines)


def _dedupe_consecutive(text):
    out, prev = [], None
    for ln in text.splitlines():
        s = ln.rstrip()
        if s and s == prev: continue
        out.append(ln)
        if s: prev = s
    return "\n".join(out)


def build_markdown(doc, clean_pages):
    heading_map = _infer_heading_levels(doc.get("pages", []),
        CONFIG["markdown_heading_size_factor"], CONFIG["markdown_max_heading_level"])
    parts = []
    title = (doc.get("title") or "").strip()
    if title: parts.append(f"# {title}\n")
    for p in clean_pages:
        body = p["text"]
        if not body: continue
        lines = []
        for ln in body.splitlines():
            stripped = ln.strip()
            if not stripped: lines.append(""); continue
            if stripped.startswith("## "):
                key = stripped[3:].strip().lower()
                lvl = heading_map.get(key)
                lines.append(("#"*lvl)+" "+stripped[3:].strip() if lvl else stripped)
            else:
                lvl = heading_map.get(stripped.lower())
                if lvl and 3 <= len(stripped) <= 120:
                    lines.append(("#"*lvl)+" "+stripped)
                else:
                    lines.append(ln)
        page_md = "\n".join(lines).strip()
        if doc.get("source_type") == "pdf" and doc.get("n_pages", 1) > 1:
            parts.append(f"<!-- page: {p['page']} -->")
        parts.append(page_md)
    md = "\n\n".join(p for p in parts if p).strip() + "\n"
    md = _dedupe_consecutive(md)
    md = re.sub(r"\n{3,}", "\n\n", md)
    if CONFIG["markdown_front_matter"]:
        md = _front_matter(doc) + md
    return md


def run_cleaning(new_rows):
    cleaned_docs = []
    for row in tqdm(new_rows, desc="Cleaning+MD", unit="doc"):
        raw_file = DIRS["raw"] / f"{row['id']}.json"
        if not raw_file.exists(): continue
        json_out = DIRS["cleaned"] / f"{row['id']}.json"
        txt_out  = DIRS["cleaned"] / f"{row['id']}.txt"
        md_out   = DIRS["cleaned"] / f"{row['id']}.md"
        if json_out.exists() and txt_out.exists() and md_out.exists():
            cleaned_docs.append(read_json(json_out)); continue
        doc = read_json(raw_file)
        if not doc: continue
        repeated = _detect_repeated_lines(doc["pages"])
        clean_pages, kept = [], 0
        for p in doc["pages"]:
            ct = clean_text(p["text"], repeated)
            kept += len(ct)
            clean_pages.append({"page": p["page"], "text": ct,
                                "method": p.get("method", "text")})
        full_text = "\n\n".join(p["text"] for p in clean_pages if p["text"])
        md_text = build_markdown(doc, clean_pages)
        out_doc = {
            "id": doc["id"], "source": doc["source"], "url": doc.get("url"),
            "title": doc["title"], "category": doc["category"],
            "source_type": doc["source_type"], "n_pages": doc["n_pages"],
            "raw_chars": doc["total_chars"], "clean_chars": kept,
            "markdown_chars": len(md_text), "markdown_file": md_out.name,
            "pages": clean_pages,
            "cleaned_at": datetime.now().isoformat(timespec="seconds"),
        }
        write_json(json_out, out_doc)
        write_text_atomic(txt_out, full_text)
        write_text_atomic(md_out, md_text)
        cleaned_docs.append(out_doc)
    # Also keep cleaned_manifest.json up to date
    cm_path = DIRS["cleaned"] / "cleaned_manifest.json"
    existing = {r["id"]: r for r in read_json(cm_path, default=[])}
    for d in cleaned_docs:
        existing[d["id"]] = {"id":d["id"],"title":d["title"],"category":d["category"],
            "source_type":d["source_type"],"n_pages":d["n_pages"],
            "clean_chars":d["clean_chars"],"markdown_chars":d["markdown_chars"],
            "markdown_file":d["markdown_file"]}
    write_json(cm_path, list(existing.values()))
    return cleaned_docs


# --- RUN ---
cleaned_new = run_cleaning(extracted)
log.info("Cleaned %d new documents.", len(cleaned_new))


Cleaning+MD:   0%|          | 0/19 [00:00<?, ?doc/s]

01:51:30 | INFO    | Cleaned 19 new documents.


## Cell 5 - Stage 4: Chunk new docs, APPEND to chunks.jsonl

Reads existing `chunks.jsonl` only to find the max global `chunk_index`, then
appends new lines. Does NOT rebuild the whole file.

In [5]:
# =============================================================================
# Stage 4: Chunk new docs and append to chunks.jsonl
# =============================================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter

HEADING_MARK = re.compile(r"^##\s+(.+)$", re.M)


def make_splitter():
    return RecursiveCharacterTextSplitter(
        chunk_size=CONFIG["chunk_size"],
        chunk_overlap=CONFIG["chunk_overlap"],
        separators=["\n## ", "\n\n", "\n", ". ", "? ", "! ", "; ", " ", ""],
        length_function=len, keep_separator=True,
    )


def _nearest_heading(text, upto):
    head = None
    for m in HEADING_MARK.finditer(text):
        if m.start() <= upto: head = m.group(1).strip()
        else: break
    return head


def chunk_document(doc, splitter):
    recs = []
    for page in doc["pages"]:
        text = page.get("text", "")
        if len(text.strip()) < CONFIG["min_chunk_chars"]: continue
        offset = 0
        for piece in splitter.split_text(text):
            piece_stripped = piece.strip()
            if len(piece_stripped) < CONFIG["min_chunk_chars"]:
                offset += len(piece); continue
            recs.append({
                "doc_id": doc["id"], "source": doc["source"],
                "url": doc.get("url"), "title": doc["title"],
                "category": doc["category"],
                "source_type": doc.get("source_type", "unknown"),
                "page": page["page"],
                "section": _nearest_heading(text, offset),
                "text": piece_stripped,
                "chunk_length": len(piece_stripped),
            })
            offset += len(piece)
    return recs


def _max_existing_chunk_index():
    """Return the next available global chunk_index."""
    if not CHUNKS_JSONL.exists(): return 0
    last_idx = -1
    with CHUNKS_JSONL.open("r", encoding="utf-8") as f:
        for ln in f:
            ln = ln.strip()
            if not ln: continue
            try: last_idx = max(last_idx, json.loads(ln).get("chunk_index", -1))
            except json.JSONDecodeError: continue
    return last_idx + 1


def run_chunking(new_docs):
    """Append chunks for new docs to chunks.jsonl. Returns the new records."""
    if not new_docs:
        log.info("Nothing new to chunk."); return []
    # De-dupe: don't re-chunk docs already in chunks.jsonl
    seen_doc_ids = set()
    if CHUNKS_JSONL.exists():
        with CHUNKS_JSONL.open("r", encoding="utf-8") as f:
            for ln in f:
                ln = ln.strip()
                if not ln: continue
                try: seen_doc_ids.add(json.loads(ln)["doc_id"])
                except (json.JSONDecodeError, KeyError): continue
    to_chunk = [d for d in new_docs if d["id"] not in seen_doc_ids]
    log.info("To chunk: %d (skipped %d already chunked)",
             len(to_chunk), len(new_docs) - len(to_chunk))
    if not to_chunk: return []

    splitter = make_splitter()
    global_idx = _max_existing_chunk_index()
    new_records = []
    with CHUNKS_JSONL.open("a", encoding="utf-8") as fout:
        for doc in tqdm(to_chunk, desc="Chunking new", unit="doc"):
            recs = chunk_document(doc, splitter)
            for r in recs:
                r["chunk_index"] = global_idx
                r["chunk_id"] = f"{r['doc_id']}::{global_idx}"
                fout.write(json.dumps(r, ensure_ascii=False) + "\n")
                new_records.append(r)
                global_idx += 1
    log.info("Appended %d new chunks (total now ends at index %d).",
             len(new_records), global_idx)
    return new_records


# --- RUN ---
new_chunks = run_chunking(cleaned_new)


01:51:39 | INFO    | To chunk: 19 (skipped 0 already chunked)


Chunking new:   0%|          | 0/19 [00:00<?, ?doc/s]

01:51:41 | INFO    | Appended 104 new chunks (total now ends at index 64285).


## Cell 6 - Stage 5: Embed new chunks, append vectors to FAISS

Loads the existing FAISS index, embeds new chunks with the same model used
originally, calls `index.add()`, then rewrites the index + `chunks_meta.json`.

⚠️ Requires Notebook 3 to have finished at least once so `faiss.index` exists.
If it doesn't, this cell logs a warning and stops — the new chunks remain in
`chunks.jsonl` and will be indexed when Notebook 3 next builds from scratch.

In [6]:
# =============================================================================
# Stage 5: Embed new chunks and append to FAISS
# =============================================================================
import faiss
from sentence_transformers import SentenceTransformer

_embedder = None


def get_embedder():
    global _embedder
    if _embedder is None:
        log.info("Loading embedding model: %s", CONFIG["embedding_model"])
        _embedder = SentenceTransformer(CONFIG["embedding_model"])
        log.info("Model loaded. Embedding dim: %d",
                 _embedder.get_sentence_embedding_dimension())
    return _embedder


def _prefix(texts, kind):
    if not CONFIG["use_e5_prefixes"]: return list(texts)
    tag = "query: " if kind == "query" else "passage: "
    return [tag + t for t in texts]


def embed(texts, kind="passage"):
    model = get_embedder()
    return model.encode(_prefix(texts, kind),
        batch_size=CONFIG["embedding_batch_size"],
        show_progress_bar=(len(texts) > 64),
        convert_to_numpy=True, normalize_embeddings=True).astype("float32")


def update_faiss(new_chunks):
    if not new_chunks:
        log.info("No new chunks to embed."); return
    if not (INDEX_FILE.exists() and META_FILE.exists()):
        log.warning("FAISS index not built yet — Notebook 3 hasn't finished. "
                    "New chunks are saved in chunks.jsonl and will be picked up "
                    "when Notebook 3 builds the index. Stopping Stage 5.")
        return

    info = read_json(INFO_FILE, default={})
    if info.get("embedding_model") and info["embedding_model"] != CONFIG["embedding_model"]:
        log.error("Existing index was built with %s but CONFIG uses %s — "
                  "ABORTING to prevent corrupting the index.",
                  info["embedding_model"], CONFIG["embedding_model"])
        return

    log.info("Loading existing FAISS index from %s", INDEX_FILE)
    index = faiss.read_index(str(INDEX_FILE))
    meta = read_json(META_FILE, default=[])
    log.info("Index currently has %d vectors / %d meta rows.",
             index.ntotal, len(meta))
    if index.ntotal != len(meta):
        log.error("Pre-existing mismatch: index.ntotal=%d vs len(meta)=%d. "
                  "Refuse to append.", index.ntotal, len(meta))
        return

    log.info("Embedding %d new chunks...", len(new_chunks))
    vectors = embed([c["text"] for c in new_chunks], kind="passage")
    if vectors.shape[1] != index.d:
        log.error("Embedding dim %d != index dim %d.", vectors.shape[1], index.d)
        return

    # Save a fresh shard for the delta so it's recoverable
    delta_shard = SHARD_DIR / f"delta_{datetime.now().strftime('%Y%m%d_%H%M%S')}.npy"
    with open(delta_shard, "wb") as f:
        np.save(f, vectors)
    log.info("Saved delta shard %s (%d vecs).", delta_shard.name, len(vectors))

    index.add(vectors)
    meta.extend(new_chunks)
    faiss.write_index(index, str(INDEX_FILE))
    write_json(META_FILE, meta)
    write_json(INFO_FILE, {
        **info,
        "n_vectors": int(index.ntotal),
        "last_updated_at": datetime.now().isoformat(timespec="seconds"),
    })
    log.info("FAISS updated: now %d vectors, %d meta rows.",
             index.ntotal, len(meta))


# --- RUN ---
update_faiss(new_chunks)
log.info("=" * 60)
log.info("Notebook 4 done.")
log.info("  new docs extracted: %d", len(extracted))
log.info("  new docs cleaned : %d", len(cleaned_new))
log.info("  new chunks added : %d", len(new_chunks))


01:52:21 | INFO    | Loading existing FAISS index from D:\Project\RAG_UPI\Dataset\_pipeline\index\faiss.index
01:52:24 | INFO    | Index currently has 72019 vectors / 72019 meta rows.
01:52:24 | INFO    | Embedding 104 new chunks...
01:52:24 | INFO    | Loading embedding model: intfloat/multilingual-e5-base


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\Rajih Nibras M\AppData\Local\Temp\ipykernel_30992\1689240112.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  _embedder.get_sentence_embedding_dimension())
01:52:35 | INFO    | Model loaded. Embedding dim: 768


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

01:52:52 | INFO    | Saved delta shard delta_20260628_015252.npy (104 vecs).
01:52:56 | INFO    | FAISS updated: now 72123 vectors, 72123 meta rows.
01:52:56 | INFO    | ============================================================
01:52:56 | INFO    | Notebook 4 done.
01:52:56 | INFO    |   new docs extracted: 19
01:52:56 | INFO    |   new docs cleaned : 19
01:52:56 | INFO    |   new chunks added : 104


---
### Re-running

Drop more PDFs anywhere inside `Dataset\New_Dataset\`, then re-run all cells.
Anything already processed is detected by stable `doc_id` and skipped at every
stage.

To remove a document from the corpus, you currently need to:
1. Delete its `raw/<id>.json`, `cleaned/<id>.{json,txt,md}` files
2. Edit `manifest.json` to remove the row
3. Rebuild `chunks.jsonl` from `cleaned/` (or use Notebook 2)
4. Rebuild the FAISS index from scratch (Notebook 3 with `force=True`)

Deletion isn't supported incrementally because FAISS `IndexFlatIP` doesn't have
stable row IDs — removing one vector would shift every higher index.
